# India Economic Early-Warning System
## Machine Learning for Detecting Upcoming Economic Slowdowns

**Portfolio Project: AI and Economics**

This notebook reproduces the complete, finalized pipeline:

1. Load and validate the raw quarterly macroeconomic dataset
2. Construct year-over-year (YoY) GDP growth
3. Define the primary **C1 slowdown target**
4. Engineer a small, economically motivated feature set
5. Build the ML-ready dataset (70 observations)
6. Evaluate three models under **5-fold expanding-window chronological validation**
7. Generate out-of-fold predictions
8. Perform a robustness check with an alternative target definition

**Primary finding (verified):**  
A regularized logistic regression that uses standard macroeconomic indicators outperforms a pure calendar-quarter baseline (mean ROC-AUC **0.735 ± 0.249** vs **0.568 ± 0.208**).

All results are obtained with strict temporal ordering — no random shuffling and no future information in predictors.


## 2. Import Libraries

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    balanced_accuracy_score, confusion_matrix,
    roc_curve
)

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
import sklearn
print('scikit-learn:', sklearn.__version__)


## 3. Load and Validate Raw Data

Source file: `economic_data.csv`  
79 quarterly observations, 2004-04-01 → 2023-10-01.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"

raw = pd.read_csv(DATA_DIR / "economic_data.csv")
raw['date'] = pd.to_datetime(raw['date'])
raw = raw.sort_values('date').reset_index(drop=True)

print('Rows:', len(raw))
print('Date range:', raw['date'].min().date(), '->', raw['date'].max().date())
print('Duplicate dates:', raw['date'].duplicated().sum())
print('Missing values:')
print(raw.isna().sum())
print('Chronological:', raw['date'].is_monotonic_increasing)
print('Months present:', sorted(raw['date'].dt.month.unique().tolist()))
print()
print(raw.head(3))
print('...')
print(raw.tail(3))


## 4. Construct YoY GDP Growth

$$
\text{yoy\_gdp\_growth}_t = \left(\frac{\text{real\_gdp}_t}{\text{real\_gdp}_{t-4}} - 1\right) \times 100
$$


In [ ]:
raw['yoy_gdp_growth'] = ((raw['real_gdp'] / raw['real_gdp'].shift(4)) - 1) * 100
raw['qoq_gdp_growth'] = ((raw['real_gdp'] / raw['real_gdp'].shift(1)) - 1) * 100  # kept for reference only
print(raw[['date', 'real_gdp', 'yoy_gdp_growth']].dropna().head(6))


## 5. Define the C1 Slowdown Target

**Definition (final):**

$$
\text{target}_t = 1 \quad \text{if} \quad \text{yoy\_gdp\_growth}_{t+1} \le \text{yoy\_gdp\_growth}_t - 1.0
$$

Otherwise \(\text{target}_t = 0\).

The target deliberately looks one quarter ahead. All predictor features use only information available at time \(t\) or earlier.


In [ ]:
raw['next_yoy'] = raw['yoy_gdp_growth'].shift(-1)
raw['target'] = np.where(
    raw['next_yoy'].isna() | raw['yoy_gdp_growth'].isna(),
    np.nan,
    (raw['next_yoy'] <= raw['yoy_gdp_growth'] - 1.0).astype(float)
)

# Quick alignment check
print(raw[['date', 'yoy_gdp_growth', 'next_yoy', 'target']].dropna().head(8))


## 6. Feature Engineering

- QoQ growth of exports and imports
- Lag-1 of key series
- Lag-4 of YoY GDP growth (seasonal reference)
- Calendar quarter (1–4) as a seasonal *control*, not the main signal


In [ ]:
raw['exports_growth'] = ((raw['exports'] / raw['exports'].shift(1)) - 1) * 100
raw['imports_growth'] = ((raw['imports'] / raw['imports'].shift(1)) - 1) * 100

raw['yoy_gdp_growth_lag1'] = raw['yoy_gdp_growth'].shift(1)
raw['yoy_gdp_growth_lag4'] = raw['yoy_gdp_growth'].shift(4)
raw['inflation_lag1'] = raw['inflation'].shift(1)
raw['industrial_production_lag1'] = raw['industrial_production'].shift(1)
raw['central_bank_rate_lag1'] = raw['central_bank_rate'].shift(1)
raw['exports_growth_lag1'] = raw['exports_growth'].shift(1)
raw['imports_growth_lag1'] = raw['imports_growth'].shift(1)
raw['quarter'] = raw['date'].dt.quarter

print('Feature engineering complete.')


## 7. Build ML-Ready Dataset

Drop rows that lack required lags or the next-quarter target.  
Expected final shape: **70 rows × 16 columns**, 2006-04-01 → 2023-07-01, target 0/1 = 46/24.


In [ ]:
final_cols = [
    'date', 'quarter',
    'yoy_gdp_growth', 'yoy_gdp_growth_lag1', 'yoy_gdp_growth_lag4',
    'inflation', 'inflation_lag1',
    'industrial_production', 'industrial_production_lag1',
    'central_bank_rate', 'central_bank_rate_lag1',
    'exports_growth', 'exports_growth_lag1',
    'imports_growth', 'imports_growth_lag1',
    'target'
]

ml = raw[final_cols].dropna().reset_index(drop=True)

print('Shape:', ml.shape)
print('Date range:', ml['date'].min().date(), '->', ml['date'].max().date())
print('Missing:', ml.isna().sum().sum())
print('Duplicate dates:', ml['date'].duplicated().sum())
print('Target distribution:')
print(ml['target'].value_counts())
print()
ml.to_csv(DATA_DIR / "ml_ready_economic_slowdown.csv", index=False)
print('Saved: ml_ready_economic_slowdown.csv')


## 8. Check for Leakage

- All growth rates and lags at time \(t\) use data ≤ \(t\).
- The only future information is the *target* (next-quarter YoY growth), which is the outcome being predicted.
- `quarter` is known at the beginning of the quarter.


In [ ]:
print('Leakage safeguards:')
print('- yoy_gdp_growth, inflation, industrial_production, central_bank_rate,')
print('  exports_growth, imports_growth  → values at t')
print('- *_lag1, yoy_gdp_growth_lag4      → values at t-1 or t-4')
print('- target                          → depends on t+1 (intentional)')
print()
print('No future macro variables appear as predictors.')


## 9. Chronological Cross-Validation

`TimeSeriesSplit(n_splits=5)` — expanding window, no shuffling.


In [ ]:
y = ml['target'].astype(int).values
dates = ml['date'].values

econ_feats = [
    'yoy_gdp_growth', 'yoy_gdp_growth_lag1', 'yoy_gdp_growth_lag4',
    'inflation', 'inflation_lag1',
    'industrial_production', 'industrial_production_lag1',
    'central_bank_rate', 'central_bank_rate_lag1',
    'exports_growth', 'exports_growth_lag1',
    'imports_growth', 'imports_growth_lag1',
    'quarter'
]
quarter_feats = ['quarter']

tscv = TimeSeriesSplit(n_splits=5)

print('Fold structure:')
for i, (tr, va) in enumerate(tscv.split(ml)):
    print(f'  Fold {i+1}: train n={len(tr)} '
          f'[{pd.Timestamp(dates[tr[0]]).date()} -> {pd.Timestamp(dates[tr[-1]]).date()}], '
          f'val n={len(va)} '
          f'[{pd.Timestamp(dates[va[0]]).date()} -> {pd.Timestamp(dates[va[-1]]).date()}], '
          f'pos={int(y[va].sum())}')


## 10–12. Model Evaluation Helper

In [ ]:
def evaluate_model(X, y, model_type='lr'):
    '''Expanding-window evaluation. Scaler fit on train only.'''
    fold_metrics = []
    all_true, all_prob, all_pred = [], [], []
    
    for fold, (tr_idx, va_idx) in enumerate(tscv.split(X)):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_va_s = scaler.transform(X_va)
        
        if model_type == 'lr':
            clf = LogisticRegression(
                penalty='l2', C=1.0, class_weight='balanced',
                max_iter=2000, random_state=RANDOM_STATE, solver='lbfgs'
            )
        else:  # rf
            clf = RandomForestClassifier(
                n_estimators=100, max_depth=3, min_samples_leaf=5,
                class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
            )
        
        clf.fit(X_tr_s, y_tr)
        prob = clf.predict_proba(X_va_s)[:, 1]
        pred = (prob >= 0.5).astype(int)
        
        n_cls = len(np.unique(y_va))
        roc = roc_auc_score(y_va, prob) if n_cls > 1 else np.nan
        pr  = average_precision_score(y_va, prob) if n_cls > 1 else np.nan
        
        fold_metrics.append({
            'fold': fold+1,
            'roc': roc, 'pr': pr,
            'prec': precision_score(y_va, pred, zero_division=0),
            'rec': recall_score(y_va, pred, zero_division=0),
            'f1': f1_score(y_va, pred, zero_division=0),
            'bal': balanced_accuracy_score(y_va, pred)
        })
        all_true.extend(y_va)
        all_prob.extend(prob)
        all_pred.extend(pred)
    
    def agg(key):
        vals = [m[key] for m in fold_metrics if not np.isnan(m[key])]
        return np.mean(vals), np.std(vals)
    
    summary = {k: agg(k) for k in ['roc','pr','prec','rec','f1','bal']}
    cm = confusion_matrix(all_true, all_pred, labels=[0,1])
    return fold_metrics, summary, cm, np.array(all_true), np.array(all_prob), np.array(all_pred)


## 10. Quarter-only Baseline

In [ ]:
print('=== Model A: Quarter-only Logistic Regression ===')
folds_A, sum_A, cm_A, _, _, _ = evaluate_model(ml[quarter_feats], y, 'lr')
for k, (m, s) in sum_A.items():
    print(f'  {k}: {m:.4f} ± {s:.4f}')
print('Pooled CM:\n', cm_A)


## 11. Economic Logistic Regression (Primary Model)

In [ ]:
print('=== Model B: Economic Logistic Regression ===')
folds_B, sum_B, cm_B, yt_B, yp_B, ypr_B = evaluate_model(ml[econ_feats], y, 'lr')
for k, (m, s) in sum_B.items():
    print(f'  {k}: {m:.4f} ± {s:.4f}')
print('Pooled CM:\n', cm_B)
print()
print('Fold ROC-AUCs:', [round(f['roc'], 4) for f in folds_B])


## 12. Economic Random Forest

In [ ]:
print('=== Model C: Economic Random Forest ===')
folds_C, sum_C, cm_C, _, _, _ = evaluate_model(ml[econ_feats], y, 'rf')
for k, (m, s) in sum_C.items():
    print(f'  {k}: {m:.4f} ± {s:.4f}')
print('Pooled CM:\n', cm_C)


## 13. Model Evaluation Summary

In [ ]:
results = pd.DataFrame({
    'Model': ['Quarter-only LR', 'Economic LR', 'Economic RF'],
    'ROC-AUC': [f"{sum_A['roc'][0]:.3f} ± {sum_A['roc'][1]:.3f}",
                f"{sum_B['roc'][0]:.3f} ± {sum_B['roc'][1]:.3f}",
                f"{sum_C['roc'][0]:.3f} ± {sum_C['roc'][1]:.3f}"],
    'PR-AUC':  [f"{sum_A['pr'][0]:.3f} ± {sum_A['pr'][1]:.3f}",
                f"{sum_B['pr'][0]:.3f} ± {sum_B['pr'][1]:.3f}",
                f"{sum_C['pr'][0]:.3f} ± {sum_C['pr'][1]:.3f}"],
    'Precision': [f"{sum_A['prec'][0]:.3f} ± {sum_A['prec'][1]:.3f}",
                  f"{sum_B['prec'][0]:.3f} ± {sum_B['prec'][1]:.3f}",
                  f"{sum_C['prec'][0]:.3f} ± {sum_C['prec'][1]:.3f}"],
    'Recall': [f"{sum_A['rec'][0]:.3f} ± {sum_A['rec'][1]:.3f}",
               f"{sum_B['rec'][0]:.3f} ± {sum_B['rec'][1]:.3f}",
               f"{sum_C['rec'][0]:.3f} ± {sum_C['rec'][1]:.3f}"],
    'F1': [f"{sum_A['f1'][0]:.3f} ± {sum_A['f1'][1]:.3f}",
           f"{sum_B['f1'][0]:.3f} ± {sum_B['f1'][1]:.3f}",
           f"{sum_C['f1'][0]:.3f} ± {sum_C['f1'][1]:.3f}"],
    'Balanced Acc': [f"{sum_A['bal'][0]:.3f} ± {sum_A['bal'][1]:.3f}",
                     f"{sum_B['bal'][0]:.3f} ± {sum_B['bal'][1]:.3f}",
                     f"{sum_C['bal'][0]:.3f} ± {sum_C['bal'][1]:.3f}"]
})
print(results.to_string(index=False))
print()
print('Primary model (Economic LR) outperforms the calendar baseline.')


## 14. Out-of-Fold Predictions (Economic LR)

Generate the 55 OOF predictions used for the timeline and confusion-matrix figures.


In [ ]:
oof_rows = []
for fold_idx, (tr_idx, va_idx) in enumerate(tscv.split(ml)):
    X_tr = ml[econ_feats].iloc[tr_idx]
    X_va = ml[econ_feats].iloc[va_idx]
    y_tr = y[tr_idx]
    
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_va_s = scaler.transform(X_va)
    
    clf = LogisticRegression(penalty='l2', C=1.0, class_weight='balanced',
                             max_iter=2000, random_state=RANDOM_STATE, solver='lbfgs')
    clf.fit(X_tr_s, y_tr)
    prob = clf.predict_proba(X_va_s)[:, 1]
    pred = (prob >= 0.5).astype(int)
    
    for j, idx in enumerate(va_idx):
        oof_rows.append({
            'date': dates[idx],
            'actual_target': int(y[idx]),
            'predicted_probability': float(prob[j]),
            'predicted_class': int(pred[j]),
            'fold': fold_idx + 1
        })

oof = pd.DataFrame(oof_rows)
oof['date'] = pd.to_datetime(oof['date'])
oof = oof.sort_values('date').reset_index(drop=True)
oof.to_csv(RESULTS_DIR / "c1_economic_lr_oof_predictions_sklearn.csv", index=False)

print('OOF observations:', len(oof))
print('Fold ROC-AUCs:', [round(f['roc'], 4) for f in folds_B])
print(f'Mean ± std: {sum_B["roc"][0]:.4f} ± {sum_B["roc"][1]:.4f}')
print('Pooled ROC-AUC:', round(roc_auc_score(oof['actual_target'], oof['predicted_probability']), 4))
print('Pooled PR-AUC:', round(average_precision_score(oof['actual_target'], oof['predicted_probability']), 4))
print('Confusion matrix:')
print(confusion_matrix(oof['actual_target'], oof['predicted_class'], labels=[0,1]))
print('Saved: c1_economic_lr_oof_predictions_sklearn.csv')


## 15. Robustness Check — Alternative Target B-5%

Alternative definition: `target = 1` if next-quarter YoY GDP growth < 5 %.

Same features, same chronological folds, same models.


In [ ]:
# Rebuild temporary B-5% target on the same feature matrix
raw2 = pd.read_csv(DATA_DIR / "economic_data.csv")
raw2['date'] = pd.to_datetime(raw2['date'])
raw2 = raw2.sort_values('date').reset_index(drop=True)
raw2['yoy'] = ((raw2['real_gdp']/raw2['real_gdp'].shift(4))-1)*100
raw2['next_yoy'] = raw2['yoy'].shift(-1)
raw2['target_b5'] = np.where(raw2['next_yoy'].isna()|raw2['yoy'].isna(), np.nan,
                             (raw2['next_yoy'] < 5.0).astype(float))

# Align with ml dates
ml_b5 = ml[['date']].merge(
    raw2[['date','target_b5']], on='date', how='left'
)
y_b5 = ml_b5['target_b5'].astype(int).values

print('B-5% positive rate: {:.1f}%'.format(100*y_b5.mean()))

_, sum_B5_econ, _, _, _, _ = evaluate_model(ml[econ_feats], y_b5, 'lr')
_, sum_B5_q, _, _, _, _ = evaluate_model(ml[quarter_feats], y_b5, 'lr')

print('B-5% Economic LR  ROC-AUC: {:.3f} ± {:.3f}'.format(*sum_B5_econ['roc']))
print('B-5% Quarter-only ROC-AUC: {:.3f} ± {:.3f}'.format(*sum_B5_q['roc']))
print()
print('Economic model still outperforms the calendar baseline under the alternative target.')


## 17. Limitations

- Only **70** usable observations; each validation fold contains just **11** observations. ROC-AUC estimates therefore exhibit substantial fold-to-fold variation.
- The dataset does not contain historical data-release timestamps/vintages. The analysis assumes that current-quarter macro variables are known when a prediction is formed.
- The model identifies statistical associations; it does **not** establish causality.
- Results should be interpreted as evidence of *feasibility*, not as a production forecasting system or an official recession predictor.
- The C1 target captures growth *deterioration*, not necessarily absolute contraction or an official recession definition.


## 18. Conclusion

Using a redesigned target based on YoY GDP growth deterioration, a simple regularized logistic regression that incorporates standard macroeconomic indicators outperformed a calendar-quarter-only baseline under chronological validation (mean ROC-AUC **0.735 ± 0.249** versus **0.568 ± 0.208**).  

The same ordering holds under an alternative absolute-threshold target (B-5%).  

The results demonstrate the feasibility of using these indicators to flag periods of potential growth slowdown, while highlighting the limitations imposed by a short quarterly sample and data-release timing.


## Reproducibility

In [ ]:
print('=== Environment ===')
print('Python:', sys.version)
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('scikit-learn:', sklearn.__version__)
import matplotlib
print('matplotlib:', matplotlib.__version__)
print()
print('Random seed:', RANDOM_STATE)
print('Notebook runs top-to-bottom with relative paths only.')
print('No external APIs or hidden state required.')
